## ResNet

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
import torch
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.45, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder('../data/CitrusUAT_split_images/train', transform=transform)
val_dataset = ImageFolder('../data/CitrusUAT_split_images/val', transform=transform)
test_dataset = ImageFolder('../data/CitrusUAT_split_images/test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)

        # 2. Log Metrics to WandB
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}')


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, test_loader, device):
    # Initialize dictionaries to store correct and total predictions
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    # Set the model to evaluation mode
    model.eval()

    # Track the ground truth labels and predictions
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move the inputs and labels to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Collect predictions and labels for metric calculations
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # Update the correct and total predictions
            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    # Calculate accuracy per class
    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname] if total_pred[classname] > 0 else 0
                          for classname in test_loader.dataset.classes}

    # Calculate overall accuracy
    overall_accuracy = accuracy_score(all_labels, all_preds)


    # Print the evaluation results
    print("Accuracy per class:")
    for classname, accuracy in accuracy_per_class.items():
        print(f"{classname}: {accuracy:.4f}")

    print()
    print(f"Overall Accuracy: {overall_accuracy:.4f}")

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
from PIL import Image
from sklearn.metrics import f1_score
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor
from tqdm import tqdm
import wandb

# RESNET -----------------------
resnet = models.resnet50(pretrained=True)
num_classes = 12
resnet.fc = torch.nn.Linear(resnet.fc.in_features, num_classes)

# Define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(resnet.fc.parameters(), lr=0.001, momentum=0.9)

# 1. Login and Initialize
wandb.login()

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="icl_hlbdetection",
    config={
        "learning_rate": 0.001,
        "architecture": "resnet",
        "dataset": "CitrusUAT",
        "epochs": 1,
        "batch_size": 32
    }
)

# Run training
model = resnet.to(device)
train(model, train_loader, val_loader, criterion, optimizer, num_epochs=1)

# 3. Close the WandB run
run.finish()

evaluate_model(model, test_loader, device)

## VGG-19

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor
from tqdm import tqdm
import wandb

# VGG19 -----------------------
# Model setup
vgg19 = models.vgg19(pretrained=True)
num_classes = len(train_dataset.classes)
vgg19.classifier[6] = torch.nn.Linear(4096, num_classes)
model = vgg19.to(device)

# Define loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# 1. Login and Initialize
wandb.login()

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="icl_hlbdetection",
    config={
        "learning_rate": 0.001,
        "architecture": "vgg19",
        "dataset": "CitrusUAT",
        "epochs": 15,
        "batch_size": 32
    }
)

# Run training
model = vgg19.to(device)
train(model, train_loader, val_loader, criterion, optimizer, num_epochs=15)

# 3. Close the WandB run
run.finish()

evaluate_model(model, test_loader, device)

## CLIP

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
from PIL import Image
from sklearn.metrics import f1_score
from transformers import CLIPProcessor, CLIPModel

# load model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# split 
train_images = []
train_labels = []
train_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "train")
for root, dirs, files in os.walk(train_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        # imgs = [{"image": file, "label": label} for file in files]
        label = root.replace(train_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        train_images.extend([os.path.join(root, file) for file in files])
        train_labels.extend([label] * len(files))

val_images = []
val_labels = []
val_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "val")
for root, dirs, files in os.walk(val_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(val_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        val_images.extend([os.path.join(root, file) for file in files])
        val_labels.extend([label] * len(files))

test_images = []
test_labels = []
test_data_path = os.path.join("..", "data", "CitrusUAT_split_images", "test")
for root, dirs, files in os.walk(test_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(test_data_path, "").replace("_", " ")
        label = label.replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency")
        test_images.extend([os.path.join(root, file) for file in files])
        test_labels.extend([label] * len(files))

# labels = [label.replace(train_data_path, "").replace("_", " ").replace("Fe", "Iron Deficiency").replace("HLB", "Huanglongbing").replace("Mg", "Magnesium Deficiency").replace("Mn", "Manganese Deficiency").replace("N", "Nitrogen Deficiency").replace("Zn", "Zinc Deficiency") for label in labels]

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
clip_model = clip_model.to(device)
clip_model.float()

class_prompts = sorted(list(set(train_labels)))
class_to_idx = {t: i for i, t in enumerate(class_prompts)}

with torch.no_grad():
    text_inputs = clip_processor(
        text=class_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    # text_outputs = clip_model.text_model(
    #     input_ids=text_inputs["input_ids"],
    #     attention_mask=text_inputs["attention_mask"],
    #     return_dict=True
    # )
    text_feats = clip_model.get_text_features(**text_inputs)
    if not torch.is_tensor(text_feats):
        text_feats = text_feats.pooler_output

    class_text_embeds = F.normalize(text_feats, dim=-1)

# create dataset
class clipdataset():
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        return image, label

def collate_fn(batch):
    images, texts = zip(*batch)
    inputs = clip_processor(
        text=list(texts),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    inputs["raw_texts"] = list(texts)
    return inputs

clip_train_dataloader = DataLoader(clipdataset(train_images, train_labels), batch_size=32, shuffle=True, collate_fn=collate_fn)

clip_val_dataloader = DataLoader(clipdataset(val_images, val_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

clip_test_dataloader = DataLoader(clipdataset(test_images, test_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

# loss function + num of correct guesses
def clip_criterion(image_embeds, text_embeds, logit_scale):
    # normalize
    image_embeds = F.normalize(image_embeds, p=2, dim=-1)
    text_embeds = F.normalize(text_embeds, p=2, dim=-1)

    logits_per_image = logit_scale * (image_embeds @ text_embeds.T)
    logits_per_text = logits_per_image.T

    batch_size = image_embeds.size(0)
    labels = torch.arange(batch_size, device=image_embeds.device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)

    return (loss_i + loss_t) / 2

lr = 1e-5
optimizer = optim.Adam(clip_model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.01)

num_epochs = 50
# import wandb

# run = wandb.init(
#     entity="rchan192-university-of-california-riverside",
#     project="my-awesome-project",
#     config={
#         "learning_rate": lr,
#         "architecture": "CLIP",
#         "dataset": "CitrusUAT",
#         "epochs": num_epochs,
#     },
# )

# train
def clip_train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        
        all_preds = []
        all_targets = []

        for batch in train_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(train_loader.dataset)
        avg_acc = total_correct / len(train_loader.dataset)

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()
        f1 = f1_score(all_targets, all_preds, average='macro')

        # validation set
        model.eval()
        total_loss_v = 0
        total_correct_v = 0

        all_preds_v = []
        all_targets_v = []
        
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(device)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
            
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_loss=False
                )

                image_embeds = outputs.image_embeds
                text_embeds = outputs.text_embeds

                logit_scale = model.logit_scale.exp().clamp(1, 100)
                loss = criterion(image_embeds, text_embeds, logit_scale)

                class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
                preds = class_logits.argmax(dim=-1)
                targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
                correct = (preds == targets).float()

                total_loss_v += loss.item() * len(pixel_values)
                total_correct_v += correct.sum().item()
                
                all_preds_v.append(preds.detach().cpu())
                all_targets_v.append(targets.detach().cpu())
        avg_loss_v = total_loss_v / len(val_loader.dataset)
        avg_acc_v = total_correct_v / len(val_loader.dataset)

        all_preds_v = torch.cat(all_preds_v).numpy()
        all_targets_v = torch.cat(all_targets_v).numpy()
        f1_v = f1_score(all_targets_v, all_preds_v, average='macro')
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}, Training Accuracy: {avg_acc:.4f}, Training F1 Score: {f1:.4f}, Validation Loss: {avg_loss_v:.4f}, Validation Accuracy: {avg_acc_v:.4f}, Validation F1 Score: {f1_v:.4f}")
        # run.log({"train loss": avg_loss,"train acc": avg_acc, "train f1 score": f1, "val loss": avg_loss_v, "val acc": avg_acc_v, "val f1 score": f1_v})

def clip_evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0

    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
        
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())
    avg_loss = total_loss / len(test_loader.dataset)
    avg_acc = total_correct / len(test_loader.dataset)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Final Average Loss: {avg_loss:.4f}, Final Average Accuracy: {avg_acc:.4f}, Final F1 Score: {f1:.4f}")

clip_train(clip_model, clip_train_dataloader, clip_val_dataloader, clip_criterion, optimizer, num_epochs, device)
clip_evaluate(clip_model, clip_test_dataloader, clip_criterion, device)

# run.finish()

## DINOV2

In [ ]:
import torch

dinov2_vitb14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')

## DINOV3

In [ ]:
from transformers import AutoModel, AutoImageProcessor

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
)